In [2]:
# Faster export: translate unique text once + emoji removal + cache + robust column detection

# If needed, run once: %pip install deep-translator

import json
import re
import time
from pathlib import Path

import pandas as pd

try:
    from deep_translator import GoogleTranslator
except ImportError as e:
    raise ImportError("Install first with: %pip install deep-translator") from e

DATA_PATH = Path("data/data_prices_cleaned.csv")
OUTPUT_CSV = Path("data/tayara_en_noemoji.csv")
CACHE_PATH = Path("data/translation_cache_en.json")

BATCH_SIZE = 50
MAX_RETRIES = 2
SLEEP_BETWEEN_BATCHES_SEC = 0.2

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing input file: {DATA_PATH.resolve()}")

df = pd.read_csv(
    DATA_PATH,
    encoding="utf-8",
    engine="python",
    on_bad_lines="skip",
)

EMOJI_RE = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map symbols
    "\U0001F1E0-\U0001F1FF"  # flags
    "\U00002700-\U000027BF"  # dingbats
    "\U000024C2-\U0001F251"
    "\U0001F900-\U0001F9FF"
    "\U0001FA70-\U0001FAFF"
    "]+",
    flags=re.UNICODE,
)

ASCII_RE = re.compile(r"^[\x00-\x7F]+$")


def remove_emojis(text):
    if pd.isna(text):
        return ""
    return EMOJI_RE.sub("", str(text)).strip()


def looks_already_english(text: str) -> bool:
    # Fast heuristic: plain ASCII text usually doesn't need translation.
    return bool(text) and bool(ASCII_RE.match(text))


def load_cache(path: Path) -> dict:
    if path.exists():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}


def save_cache(path: Path, cache: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(cache, ensure_ascii=False), encoding="utf-8")


def first_existing_column(frame: pd.DataFrame, candidates):
    for c in candidates:
        if c in frame.columns:
            return c
    return None


def translate_unique_to_en(unique_texts, cache: dict, batch_size=BATCH_SIZE):
    translator = GoogleTranslator(source="auto", target="en")

    to_translate = []
    for t in unique_texts:
        if not t:
            continue
        if t in cache:
            continue
        if looks_already_english(t):
            cache[t] = t
        else:
            to_translate.append(t)

    total = len(to_translate)
    print(f"Unique candidates to translate: {total}")

    started = time.time()
    for i in range(0, total, batch_size):
        batch = to_translate[i:i + batch_size]

        translated = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                translated = translator.translate_batch(batch)
                if isinstance(translated, str):
                    translated = [translated]
                if translated is None:
                    translated = []
                break
            except Exception:
                if attempt == MAX_RETRIES:
                    translated = []
                else:
                    time.sleep(0.8 * attempt)

        if translated:
            for src, dst in zip(batch, translated):
                clean_dst = remove_emojis(dst) if isinstance(dst, str) else ""
                cache[src] = clean_dst if clean_dst else src

        if len(translated) < len(batch):
            for src in batch[len(translated):]:
                cache[src] = src

        if ((i // batch_size) + 1) % 10 == 0:
            save_cache(CACHE_PATH, cache)

        done = min(i + len(batch), total)
        elapsed = time.time() - started
        rate = done / elapsed if elapsed > 0 else 0
        eta = (total - done) / rate if rate > 0 else 0
        print(f"Progress: {done}/{total} | {rate:.1f} txt/s | ETA ~ {eta/60:.1f} min")

        if SLEEP_BETWEEN_BATCHES_SEC:
            time.sleep(SLEEP_BETWEEN_BATCHES_SEC)

    return cache


translated_df = df.copy()

# Handle either singular or plural source columns
title_src = first_existing_column(translated_df, ["title", "titles"])
desc_src = first_existing_column(translated_df, ["description", "descriptions"])

print(f"Using title source column: {title_src}")
print(f"Using description source column: {desc_src}")

if title_src is None:
    translated_df["title_original"] = ""
else:
    translated_df["title_original"] = translated_df[title_src]

if desc_src is None:
    translated_df["description_original"] = ""
else:
    translated_df["description_original"] = translated_df[desc_src]

translated_df["title_clean"] = translated_df["title_original"].map(remove_emojis)
translated_df["description_clean"] = translated_df["description_original"].map(remove_emojis)

all_unique = pd.concat([
    translated_df["title_clean"],
    translated_df["description_clean"],
]).fillna("").astype(str).str.strip().unique().tolist()

cache = load_cache(CACHE_PATH)
cache = translate_unique_to_en(all_unique, cache, batch_size=BATCH_SIZE)
save_cache(CACHE_PATH, cache)

translated_df["title_en"] = translated_df["title_clean"].map(lambda x: cache.get(x, x))
translated_df["description_en"] = translated_df["description_clean"].map(lambda x: cache.get(x, x))

# Standardized output columns
translated_df["title"] = translated_df["title_en"]
translated_df["description"] = translated_df["description_en"]

translated_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"Saved translated file to: {OUTPUT_CSV.resolve()}")
display(translated_df[["title_original", "title", "description_original", "description"]].head(3))

Using title source column: titles
Using description source column: descriptions
Unique candidates to translate: 12900
Progress: 50/12900 | 1.4 txt/s | ETA ~ 153.9 min
Progress: 100/12900 | 1.3 txt/s | ETA ~ 161.4 min
Progress: 150/12900 | 1.3 txt/s | ETA ~ 163.7 min
Progress: 200/12900 | 1.4 txt/s | ETA ~ 155.2 min
Progress: 250/12900 | 1.4 txt/s | ETA ~ 154.0 min
Progress: 300/12900 | 1.4 txt/s | ETA ~ 151.7 min
Progress: 350/12900 | 1.4 txt/s | ETA ~ 148.9 min
Progress: 400/12900 | 1.4 txt/s | ETA ~ 150.9 min
Progress: 450/12900 | 1.4 txt/s | ETA ~ 150.8 min
Progress: 500/12900 | 1.4 txt/s | ETA ~ 150.0 min
Progress: 550/12900 | 1.4 txt/s | ETA ~ 149.2 min
Progress: 600/12900 | 1.4 txt/s | ETA ~ 147.8 min
Progress: 650/12900 | 1.4 txt/s | ETA ~ 147.2 min
Progress: 700/12900 | 1.4 txt/s | ETA ~ 146.6 min
Progress: 750/12900 | 1.4 txt/s | ETA ~ 147.4 min
Progress: 800/12900 | 1.4 txt/s | ETA ~ 146.4 min
Progress: 850/12900 | 1.4 txt/s | ETA ~ 146.2 min
Progress: 900/12900 | 1.4 txt/s |

,title_original,title,description_original,description
0,A Vendre un appartement s+1 direct promoteur à...,For sale an apartment s + 1 direct developer i...,’agence MSM Immobilière vous propose à vendre ...,The MSM Immobilière agency offers you for sale...
1,Appartement S+2 à chatt meriem,Apartment S+2 in chatt meriem,🤗🤗 Opportunité à saisir 🤗🤗\nVENDO IMMOBILIERE ...,Opportunity to seize \nVENDO IMMOBILIERE Offer...
2,Villa à Sahloul 3,Villa in Sahloul 3,l'agence immobilière #HPS vous propose la vent...,the real estate agency #HPS offers you the sal...
